In [108]:
import ssl 
ssl._create_default_https_context = ssl._create_unverified_context
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import re
import nltk
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords
from tqdm import tqdm
import time

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline 
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer

nltk.download('stopwords')
stop = stopwords.words('english')



[nltk_data] Downloading package stopwords to /Users/user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [109]:
import random

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [110]:
def preprocessor(text):
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)',
                           text)
    text = (re.sub(r'[\W]+', ' ', text.lower()) +
            ' '.join(emoticons).replace('-', ''))
    return text

def tokenizer(text):
    return text.split()

porter = PorterStemmer()
def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]

def stem_and_stop_remove(text):
    tokens =  [w for w in tokenizer_porter(text) if w not in stop]
    return " ".join(tokens)


In [111]:
# 1. transform imdb data into tf-idf form.
# read in data

path = "movie_data.csv"
print("Reading CSV...")
data = pd.read_csv(path)
print("Done!")
print()

X_train = data.loc[:34999, 'review']
y_train = data.loc[:34999, 'sentiment']
X_test = data.loc[35000:, 'review']
y_test = data.loc[35000:, 'sentiment']  

# clean data of HTML formatting (taken from pg. 255) confirmed working. 
print("Beginning preprocessing:")
X_train = X_train.apply(preprocessor)
X_test = X_test.apply(preprocessor)
print("Done!")
print()

print("Beginning stemmer and stop word remover...")
X_train = X_train.apply(stem_and_stop_remove)
X_test = X_test.apply(stem_and_stop_remove)


Reading CSV...
Done!

Beginning preprocessing:
Done!

Beginning stemmer and stop word remover...


In [219]:
# Hyperparameters
input_size = 5000
hidden1 = 350
hidden2 = 100
hidden3 = 32
hidden4 = 32
output_size = 2

learning_rate = 0.0015
batch_size = 64
num_epochs = 30
weight_decay = 1e-3

In [220]:
print("running CountVectorizer...")
cv = CountVectorizer(max_features=input_size)
# this needs the data as a list
train_bag = cv.fit_transform(np.array(X_train.tolist()))
test_bag = cv.transform(np.array(X_test.tolist()))
print("Done!")
print()

# do we use the vectorizer or the transformer here? book uses the vectorizer for the log reg code. 
print("running tf-idf...")
tf_idf = TfidfTransformer(use_idf=True, norm='l2', smooth_idf=True)
train_bag = tf_idf.fit_transform(train_bag)
test_bag = tf_idf.transform(test_bag)
print("Done!")

running CountVectorizer...
Done!

running tf-idf...
Done!


In [221]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# device = torch.device("cpu")

# torch.manual_seed(1)

X_train_tensor = torch.tensor(train_bag.toarray(), dtype=torch.float32)
y_train_tensor = torch.tensor(np.array(y_train), dtype=torch.long)
X_test_tensor = torch.tensor(test_bag.toarray(), dtype=torch.float32)
y_test_tensor = torch.tensor(np.array(y_test), dtype=torch.long)

X_train_tensor = X_train_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)

joint_dataset = TensorDataset(X_train_tensor, y_train_tensor)

g = torch.Generator()
g.manual_seed(seed)

train_data = DataLoader(joint_dataset, batch_size=batch_size, shuffle=True, generator=g)


Using device: cpu


In [222]:
class FNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden1)
        self.fc2 = torch.nn.Linear(hidden1, hidden2)
        self.fc3 = torch.nn.Linear(hidden2, output_size)
        #self.fc4 = torch.nn.Linear(hidden3, hidden4)
        #self.fc5 = torch.nn.Linear(hidden4, output_size)
        #self.fc6 = torch.nn.Linear(hidden5, output_size)

    def forward(self, x):
        x = torch.nn.functional.relu(self.fc1(x))
        x = torch.nn.functional.relu(self.fc2(x))
        #x = torch.nn.functional.relu(self.fc3(x))
        #x = torch.nn.functional.relu(self.fc4(x))
        #x = torch.nn.functional.relu(self.fc5(x))
        x = self.fc3(x)
        return x

    
net = FNN().to(device)

In [223]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

nn_loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr = learning_rate, weight_decay=weight_decay)

start_time = time.time()

for i in range(num_epochs):

    net.train()   

    print("epoch", i+1, "/", num_epochs)

    num_correct = 0
    total = 0

    for (x, y) in train_data:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = net(x)
        loss = nn_loss(output, y)

        loss.backward()
        optimizer.step()

        preds = torch.argmax(output, dim=1)
        num_correct += (preds == y).sum().item()
        total += y.size(0)

    train_accuracy = num_correct / total
    print("Training Accuracy:", train_accuracy)
    print()

    net.eval()

    with torch.no_grad():

        X_test_tensor = X_test_tensor.to(device)
        y_test_tensor = y_test_tensor.to(device)

        y_pred = net(X_test_tensor)
        pred_labels = torch.argmax(y_pred, dim=1)

        accuracy = (pred_labels == y_test_tensor).float().mean().item()

    print("Test Accuracy:", accuracy)

end_time = time.time()
nn_time = end_time - start_time

print("FNN Training Time:", nn_time, "seconds")

epoch 1 / 30
Training Accuracy: 0.8567428571428571

Test Accuracy: 0.8245333433151245
epoch 2 / 30
Training Accuracy: 0.8883428571428571

Test Accuracy: 0.8390666842460632
epoch 3 / 30
Training Accuracy: 0.8964571428571428

Test Accuracy: 0.8370000123977661
epoch 4 / 30
Training Accuracy: 0.9084285714285715

Test Accuracy: 0.834933340549469
epoch 5 / 30
Training Accuracy: 0.9142285714285714

Test Accuracy: 0.8358666896820068
epoch 6 / 30
Training Accuracy: 0.9179142857142857

Test Accuracy: 0.8612666726112366
epoch 7 / 30
Training Accuracy: 0.9214

Test Accuracy: 0.8244666457176208
epoch 8 / 30


KeyboardInterrupt: 